# Part 2: Walk-Forward Experimental Pipeline

This notebook implements a clean rolling walk-forward experiment over `2010-01-01` through `2022-12-31`.

Key design choices:
- 5-year train, 1-year validation, 1-year test
- Test years `2017` through `2022`
- Target is 5-day-ahead cross-sectional return
- Portfolio rebalances every 5 trading days
- Long top 20%, short bottom 20%, dollar-neutral
- Transaction costs use the existing turnover-based method
- Model direction is fixed at `1` for every model and fold


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import json
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from IPython.display import display
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
START_DATE = "2010-01-01"
END_DATE = "2022-12-31"
TARGET_COL = "target_5d_cs"
REALIZED_RETURN_COL = "target_5d_raw"
REBALANCE_EVERY_N_DAYS = 5
LONG_SHORT_FRACTION = 0.20
TRANSACTION_COST = 0.0005
MIN_NAMES = 10

universe = [
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN", "MU", "ADI",
    "MRVL", "MCHP", "MPWR", "ON", "NXPI", "SWKS", "QRVO",
    "LSCC", "CRUS", "DIOD", "POWI", "ALGM",
    "AMAT", "LRCX", "KLAC", "TER", "ENTG", "ACLS", "AEHR",
    "ASML", "ASX", "UCTT", "FORM", "COHR",
    "TSM", "UMC", "STM", "GFS",
    "SNPS", "CDNS", "ANSS",
    "AAPL", "MSFT", "GOOGL", "META", "AMZN",
    "DELL", "HPQ", "SMCI", "HPE", "ARW", "AVT",
]

In [2]:
def make_price_panel(raw_data, requested_universe):
    frames = []
    valid_tickers = []

    for ticker in requested_universe:
        try:
            px = raw_data[ticker].copy()
        except Exception:
            continue

        if px.empty or "Close" not in px.columns or px["Close"].dropna().empty:
            continue

        keep_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in px.columns]
        px = px[keep_cols].copy()
        px["code"] = ticker
        frames.append(px.reset_index().rename(columns={"Date": "date"}))
        valid_tickers.append(ticker)

    if not frames:
        raise ValueError("No valid tickers were downloaded.")

    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["code", "date"]).reset_index(drop=True)

    price_cols = ["Open", "High", "Low", "Close"]
    df[price_cols] = df.groupby("code")[price_cols].ffill()
    df["Volume"] = df.groupby("code")["Volume"].transform(lambda x: x.fillna(0))
    df["Return"] = df.groupby("code")["Close"].pct_change()
    df = df.dropna(subset=price_cols + ["Return"]).reset_index(drop=True)
    return df, valid_tickers


def add_technical_features(df):
    df = df.sort_values(["code", "date"]).reset_index(drop=True).copy()
    g = df.groupby("code", group_keys=False)
    f = pd.DataFrame(index=df.index)

    f["open_1"] = g["Open"].shift(1)
    f["high_1"] = g["High"].shift(1)
    f["low_1"] = g["Low"].shift(1)
    f["close_1"] = g["Close"].shift(1)
    f["close_2"] = g["Close"].shift(2)
    f["volume_1"] = g["Volume"].shift(1)
    f["log_volume_1"] = np.log1p(f["volume_1"])

    f["overnight_gap_1"] = f["open_1"] / f["close_2"] - 1
    f["intraday_return_1"] = f["close_1"] / f["open_1"] - 1
    f["high_low_range_1"] = f["high_1"] / f["low_1"] - 1
    f["high_close_range_1"] = f["high_1"] / f["close_1"] - 1
    f["low_close_range_1"] = f["low_1"] / f["close_1"] - 1
    f["close_position_1"] = (f["close_1"] - f["low_1"]) / (f["high_1"] - f["low_1"]).replace(0, np.nan)
    f["candle_body_1"] = (f["close_1"] - f["open_1"]) / f["open_1"]
    f["candle_body_abs_1"] = f["candle_body_1"].abs()
    f["upper_shadow_1"] = (f["high_1"] - np.maximum(f["open_1"], f["close_1"])) / f["close_1"]
    f["lower_shadow_1"] = (np.minimum(f["open_1"], f["close_1"]) - f["low_1"]) / f["close_1"]

    for window in [1, 2, 5, 10, 20, 60, 120, 250]:
        f[f"return_{window}"] = g["Close"].transform(lambda x, w=window: (x / x.shift(w) - 1).shift(1))
    for window in [1, 5, 20, 60, 250]:
        f[f"log_return_{window}"] = g["Close"].transform(lambda x, w=window: np.log(x).diff(w).shift(1))

    f["momentum_20_minus_5"] = f["return_20"] - f["return_5"]
    f["momentum_60_minus_20"] = f["return_60"] - f["return_20"]
    f["momentum_250_minus_60"] = f["return_250"] - f["return_60"]

    for window in [5, 20, 60, 250]:
        f[f"ret_mean_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).mean().shift(1))
        f[f"ret_std_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).std().shift(1))
        f[f"sharpe_{window}"] = f[f"ret_mean_{window}"] / f[f"ret_std_{window}"]

    for window in [20, 60]:
        f[f"ret_skew_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).skew().shift(1))
        f[f"ret_kurt_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).kurt().shift(1))

    f["return_5_to_vol_20"] = f["return_5"] / f["ret_std_20"]
    f["return_20_to_vol_60"] = f["return_20"] / f["ret_std_60"]

    ma = {}
    for window in [5, 10, 20, 60, 120, 250]:
        ma[window] = g["Close"].transform(lambda x, w=window: x.rolling(w).mean().shift(1))
        f[f"close_to_ma_{window}"] = f["close_1"] / ma[window] - 1

    f["ma_5_to_20"] = ma[5] / ma[20] - 1
    f["ma_10_to_20"] = ma[10] / ma[20] - 1
    f["ma_20_to_60"] = ma[20] / ma[60] - 1
    f["ma_60_to_120"] = ma[60] / ma[120] - 1
    f["ma_60_to_250"] = ma[60] / ma[250] - 1
    f["ma_120_to_250"] = ma[120] / ma[250] - 1

    for window in [20, 60]:
        price_std = g["Close"].transform(lambda x, w=window: x.rolling(w).std().shift(1))
        f[f"bollinger_z_{window}"] = (f["close_1"] - ma[window]) / price_std
        f[f"bollinger_width_{window}"] = 4 * price_std / ma[window]

    for window in [20, 60, 250]:
        rolling_high = g["High"].transform(lambda x, w=window: x.rolling(w).max().shift(1))
        rolling_low = g["Low"].transform(lambda x, w=window: x.rolling(w).min().shift(1))
        f[f"dist_to_high_{window}"] = f["close_1"] / rolling_high - 1
        f[f"dist_to_low_{window}"] = f["close_1"] / rolling_low - 1
        f[f"range_position_{window}"] = (f["close_1"] - rolling_low) / (rolling_high - rolling_low).replace(0, np.nan)
        f[f"drawdown_{window}"] = f["close_1"] / rolling_high - 1

    prev_close = g["Close"].shift(1)
    true_range = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    tr_df = pd.DataFrame({"code": df["code"], "true_range": true_range})
    tr_g = tr_df.groupby("code", group_keys=False)
    for window in [14, 20, 60]:
        f[f"natr_{window}"] = tr_g["true_range"].transform(lambda x, w=window: x.rolling(w).mean().shift(1)) / f["close_1"]

    def rsi(x, window):
        diff = x.diff()
        gain = diff.clip(lower=0).rolling(window).mean()
        loss = (-diff.clip(upper=0)).rolling(window).mean()
        return (100 - 100 / (1 + gain / loss)).shift(1)

    f["rsi_14"] = g["Close"].transform(lambda x: rsi(x, 14))
    f["rsi_28"] = g["Close"].transform(lambda x: rsi(x, 28))
    f["ema_12_to_26"] = g["Close"].transform(lambda x: (x.ewm(span=12, adjust=False).mean() / x.ewm(span=26, adjust=False).mean() - 1).shift(1))
    f["ema_20_to_60"] = g["Close"].transform(lambda x: (x.ewm(span=20, adjust=False).mean() / x.ewm(span=60, adjust=False).mean() - 1).shift(1))
    macd_signal = g["Close"].transform(
        lambda x: ((x.ewm(span=12, adjust=False).mean() - x.ewm(span=26, adjust=False).mean()) / x.ewm(span=26, adjust=False).mean()).ewm(span=9, adjust=False).mean().shift(1)
    )
    f["macd_signal_9"] = macd_signal
    f["macd_hist"] = f["ema_12_to_26"] - f["macd_signal_9"]

    for window in [1, 5, 20]:
        f[f"volume_log_change_{window}"] = g["Volume"].transform(lambda x, w=window: np.log1p(x).diff(w).shift(1))

    log_volume = np.log1p(df["Volume"])
    lv_df = pd.DataFrame({"code": df["code"], "log_volume": log_volume})
    lv_g = lv_df.groupby("code", group_keys=False)
    for window in [20, 60]:
        lv_mean = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).mean().shift(1))
        lv_std = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).std().shift(1))
        f[f"relative_log_volume_{window}"] = f["log_volume_1"] - lv_mean
        f[f"volume_zscore_{window}"] = (f["log_volume_1"] - lv_mean) / lv_std
        f[f"volume_rank_{window}"] = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).rank(pct=True).shift(1))

    dollar_volume = df["Close"] * df["Volume"]
    dv_df = pd.DataFrame({"code": df["code"], "dollar_volume": dollar_volume})
    dv_g = dv_df.groupby("code", group_keys=False)
    for window in [1, 5, 20]:
        f[f"dollar_volume_log_change_{window}"] = dv_g["dollar_volume"].transform(lambda x, w=window: np.log1p(x).diff(w).shift(1))

    cs_base = pd.concat([df[["date"]], f], axis=1)
    for col in ["return_1", "return_5", "return_20", "return_60", "ret_std_20", "ret_std_60", "volume_zscore_20", "close_to_ma_20", "range_position_60"]:
        f[f"{col}_cs_rank"] = cs_base.groupby("date")[col].rank(pct=True)

    f["day_of_week"] = df["date"].dt.dayofweek
    f["month"] = df["date"].dt.month
    f["quarter"] = df["date"].dt.quarter
    f["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    f["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    f["is_quarter_start"] = df["date"].dt.is_quarter_start.astype(int)
    f["is_quarter_end"] = df["date"].dt.is_quarter_end.astype(int)

    out = pd.concat([df, f], axis=1)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def prepare_model_data(df):
    work = df.copy()
    work = work[(work["date"] >= START_DATE) & (work["date"] <= END_DATE)].copy()
    work["target_5d_raw"] = work.groupby("code")["Close"].shift(-5) / work["Close"] - 1
    work["target_5d_cs"] = work["target_5d_raw"] - work.groupby("date")["target_5d_raw"].transform("mean")

    base_cols = [
        "date", "code", "Open", "High", "Low", "Close", "Volume", "Return",
        "target_5d_raw", "target_5d_cs",
    ]
    feature_cols = [c for c in work.columns if c not in base_cols]
    work = work.dropna(subset=feature_cols + ["target_5d_raw", "target_5d_cs"]).copy()
    work = work.sort_values(["date", "code"]).reset_index(drop=True)
    return work, feature_cols


raw_data = yf.download(
    universe,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True,
    group_by="ticker",
    threads=True,
)

price_df, valid_tickers = make_price_panel(raw_data, universe)
feature_df = add_technical_features(price_df)
model_df, feature_cols = prepare_model_data(feature_df)

print(f"Valid tickers: {len(valid_tickers)} / {len(universe)}")
print(f"Model rows: {len(model_df):,}")
print(f"Feature count: {len(feature_cols)}")
model_df[["date", "code", "target_5d_raw", "target_5d_cs"]].head()

[**********************50%                       ]  25 of 50 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
[*********************100%***********************]  50 of 50 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANSS']: YFTzMissingError('possibly delisted; no timezone found')
/tmp/ipykernel_1058/2587922802.py:153: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f[f"{col}_cs_rank"] = cs_base.groupby("date")[col].rank(pct=True)
/tmp/ipykernel_1058/2587922802.py:153: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

Valid tickers: 49 / 50
Model rows: 136,773
Feature count: 116


,date,code,target_5d_raw,target_5d_cs
0,2011-01-03,AAPL,0.039081,0.018223
1,2011-01-03,ACLS,-0.116022,-0.136880
2,2011-01-03,ADI,-0.006587,-0.027445
3,2011-01-03,AEHR,0.118644,0.097786
4,2011-01-03,AMAT,-0.020626,-0.041484


In [3]:
def make_fold_specs(test_years=range(2017, 2023), train_years=5):
    specs = []
    for fold_number, test_year in enumerate(test_years, start=1):
        val_year = test_year - 1
        train_start_year = test_year - (train_years + 1)
        train_end_year = test_year - 2
        specs.append({
            "fold_name": f"Fold {fold_number}",
            "train_start": f"{train_start_year}-01-01",
            "train_end": f"{train_end_year}-12-31",
            "val_start": f"{val_year}-01-01",
            "val_end": f"{val_year}-12-31",
            "test_start": f"{test_year}-01-01",
            "test_end": f"{test_year}-12-31",
            "test_year": test_year,
        })
    return specs


FOLD_SPECS = make_fold_specs()
pd.DataFrame(FOLD_SPECS)


,fold_name,train_start,train_end,val_start,val_end,test_start,test_end,test_year
0,Fold 1,2011-01-01,2015-12-31,2016-01-01,2016-12-31,2017-01-01,2017-12-31,2017
1,Fold 2,2012-01-01,2016-12-31,2017-01-01,2017-12-31,2018-01-01,2018-12-31,2018
2,Fold 3,2013-01-01,2017-12-31,2018-01-01,2018-12-31,2019-01-01,2019-12-31,2019
3,Fold 4,2014-01-01,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,2020
4,Fold 5,2015-01-01,2019-12-31,2020-01-01,2020-12-31,2021-01-01,2021-12-31,2021
5,Fold 6,2016-01-01,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,2022


In [4]:
def create_time_split(df_model, spec, date_col="date"):
    df_sorted = df_model.sort_values([date_col, "code"]).copy()
    return {
        "train": df_sorted[df_sorted[date_col].between(spec["train_start"], spec["train_end"])].copy(),
        "validation": df_sorted[df_sorted[date_col].between(spec["val_start"], spec["val_end"])].copy(),
        "test": df_sorted[df_sorted[date_col].between(spec["test_start"], spec["test_end"])].copy(),
    }


def get_rebalance_dates(df_eval, every_n_days=REBALANCE_EVERY_N_DAYS):
    dates = np.array(sorted(pd.to_datetime(df_eval["date"].unique())))
    return dates[::every_n_days]


def build_long_short_weights(
    df_eval,
    signal_col,
    top_frac=LONG_SHORT_FRACTION,
    rebalance_dates=None,
    min_names=MIN_NAMES,
    date_col="date",
    code_col="code",
):
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval)

    weight_frames = []
    for date in rebalance_dates:
        cross_section = df_eval.loc[df_eval[date_col] == date, [date_col, code_col, signal_col]].dropna().copy()
        n = len(cross_section)
        if n < min_names:
            continue

        n_bucket = max(1, int(np.floor(n * top_frac)))
        cross_section = cross_section.sort_values(signal_col, ascending=False)
        longs = cross_section.head(n_bucket).copy()
        shorts = cross_section.tail(n_bucket).copy()

        longs["weight"] = 0.5 / len(longs)
        shorts["weight"] = -0.5 / len(shorts)
        weight_frames.append(pd.concat([longs, shorts], ignore_index=True)[[date_col, code_col, "weight"]])

    if not weight_frames:
        return pd.DataFrame(columns=[date_col, code_col, "weight"])
    return pd.concat(weight_frames, ignore_index=True)


def build_equal_weight_long_only_weights(df_eval, rebalance_dates=None, date_col="date", code_col="code"):
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval)

    frames = []
    for date in rebalance_dates:
        names = sorted(df_eval.loc[df_eval[date_col] == date, code_col].dropna().unique())
        if not names:
            continue
        frames.append(pd.DataFrame({date_col: date, code_col: names, "weight": 1.0 / len(names)}))

    if not frames:
        return pd.DataFrame(columns=[date_col, code_col, "weight"])
    return pd.concat(frames, ignore_index=True)


def build_random_long_short_weights(df_eval, seed, rebalance_dates=None, top_frac=LONG_SHORT_FRACTION, min_names=MIN_NAMES, date_col="date", code_col="code"):
    rng = np.random.default_rng(seed)
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval)

    frames = []
    for date in rebalance_dates:
        names = np.array(sorted(df_eval.loc[df_eval[date_col] == date, code_col].dropna().unique()))
        n = len(names)
        if n < min_names:
            continue
        n_bucket = max(1, int(np.floor(n * top_frac)))
        selected = rng.choice(names, size=2 * n_bucket, replace=False)
        longs = selected[:n_bucket]
        shorts = selected[n_bucket:]
        frames.append(pd.DataFrame({
            date_col: date,
            code_col: np.concatenate([longs, shorts]),
            "weight": np.concatenate([
                np.repeat(0.5 / n_bucket, n_bucket),
                np.repeat(-0.5 / n_bucket, n_bucket),
            ]),
        }))

    if not frames:
        return pd.DataFrame(columns=[date_col, code_col, "weight"])
    return pd.concat(frames, ignore_index=True)


def long_short_backtest(df_eval, weights, target_col=REALIZED_RETURN_COL, transaction_cost=TRANSACTION_COST, date_col="date", code_col="code"):
    df_bt = weights.merge(df_eval[[date_col, code_col, target_col]], on=[date_col, code_col], how="left")
    df_bt = df_bt.dropna(subset=[target_col, "weight"]).copy()

    rows = []
    prev_weights = pd.Series(dtype=float)
    for date, part in df_bt.groupby(date_col, sort=True):
        current_weights = part.set_index(code_col)["weight"].astype(float)
        all_names = current_weights.index.union(prev_weights.index)
        turnover = (current_weights.reindex(all_names, fill_value=0.0) - prev_weights.reindex(all_names, fill_value=0.0)).abs().sum()
        gross_return = (part["weight"] * part[target_col]).sum()
        cost = transaction_cost * turnover
        net_return = gross_return - cost

        rows.append({
            "date": pd.Timestamp(date),
            "gross_return": gross_return,
            "net_return": net_return,
            "turnover": turnover,
            "cost": cost,
            "n_positions": len(current_weights),
        })
        prev_weights = current_weights

    result = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    if result.empty:
        return result

    result["gross_equity"] = (1 + result["gross_return"]).cumprod()
    result["net_equity"] = (1 + result["net_return"]).cumprod()
    result["gross_cum_return"] = result["gross_equity"] - 1
    result["net_cum_return"] = result["net_equity"] - 1
    result["net_drawdown"] = result["net_equity"] / result["net_equity"].cummax() - 1
    return result


def backtest_metrics(backtest_result, return_col="net_return", periods_per_year=252 / REBALANCE_EVERY_N_DAYS):
    if backtest_result.empty:
        return {
            "cumulative_return": np.nan,
            "annualized_return": np.nan,
            "annualized_volatility": np.nan,
            "sharpe_ratio": np.nan,
            "max_drawdown": np.nan,
            "average_turnover": np.nan,
            "n_periods": 0,
        }

    r = backtest_result[return_col].dropna()
    if r.empty:
        return {
            "cumulative_return": np.nan,
            "annualized_return": np.nan,
            "annualized_volatility": np.nan,
            "sharpe_ratio": np.nan,
            "max_drawdown": np.nan,
            "average_turnover": np.nan,
            "n_periods": 0,
        }

    equity = (1 + r).cumprod()
    annualized_return = equity.iloc[-1] ** (periods_per_year / len(r)) - 1
    annualized_volatility = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility and annualized_volatility > 0 else np.nan
    drawdown = equity / equity.cummax() - 1

    return {
        "cumulative_return": equity.iloc[-1] - 1,
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": drawdown.min(),
        "average_turnover": backtest_result["turnover"].mean(),
        "n_periods": len(r),
    }


def evaluate_signal(df_eval, signal_col, target_col=REALIZED_RETURN_COL, rebalance_dates=None):
    weights = build_long_short_weights(df_eval, signal_col, rebalance_dates=rebalance_dates)
    bt = long_short_backtest(df_eval, weights, target_col=target_col)
    return bt, backtest_metrics(bt)

In [5]:
def make_baseline_signal_frame(df_part):
    baseline_df = df_part[["date", "code", "target_5d_raw", "target_5d_cs", "return_20", "return_60", "ret_std_20"]].copy()
    baseline_df["mom_20"] = baseline_df["return_20"]
    baseline_df["mom_60"] = baseline_df["return_60"]
    baseline_df["reversal_20"] = -baseline_df["return_20"]
    baseline_df["low_vol"] = -baseline_df["ret_std_20"]
    return baseline_df


def build_strategy_result_row(test_year, strategy_name, metrics, category):
    return {
        "test_year": test_year,
        "strategy": strategy_name,
        "category": category,
        "net_cumulative_return": metrics["cumulative_return"],
        "sharpe_ratio": metrics["sharpe_ratio"],
        "max_drawdown": metrics["max_drawdown"],
        "average_turnover": metrics["average_turnover"],
        "n_rebalance_periods": metrics["n_periods"],
    }


MODEL_SPECS = {
    "Ridge / Linear Regression": [
        {
            "label": "alpha_0.1",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=0.1)),
            ]),
        },
        {
            "label": "alpha_1.0",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=1.0)),
            ]),
        },
        {
            "label": "alpha_10.0",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=10.0)),
            ]),
        },
    ],
    "Random Forest Regressor": [
        {
            "label": "rf_200_d6",
            "estimator": RandomForestRegressor(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=20,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        },
        {
            "label": "rf_300_d8",
            "estimator": RandomForestRegressor(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        },
        {
            "label": "rf_400_none",
            "estimator": RandomForestRegressor(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=10,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        },
    ],
    "MLP Regressor": [
        {
            "label": "mlp_64x1",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(
                    hidden_layer_sizes=(64,),
                    alpha=1e-4,
                    learning_rate_init=1e-3,
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                )),
            ]),
        },
        {
            "label": "mlp_64x32",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(
                    hidden_layer_sizes=(64, 32),
                    alpha=1e-4,
                    learning_rate_init=1e-3,
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                )),
            ]),
        },
        {
            "label": "mlp_128x64",
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(
                    hidden_layer_sizes=(128, 64),
                    alpha=1e-3,
                    learning_rate_init=5e-4,
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                )),
            ]),
        },
    ],
    "LightGBM Regressor": [
        {
            "label": "lgbm_small",
            "estimator": lgb.LGBMRegressor(
                objective="regression",
                n_estimators=200,
                learning_rate=0.03,
                max_depth=3,
                num_leaves=15,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=1.0,
                reg_lambda=10.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        },
        {
            "label": "lgbm_medium",
            "estimator": lgb.LGBMRegressor(
                objective="regression",
                n_estimators=300,
                learning_rate=0.03,
                max_depth=4,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.5,
                reg_lambda=5.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        },
        {
            "label": "lgbm_stronger",
            "estimator": lgb.LGBMRegressor(
                objective="regression",
                n_estimators=400,
                learning_rate=0.02,
                max_depth=4,
                num_leaves=31,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=1.0,
                reg_lambda=10.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        },
    ],
}


In [6]:
def select_best_model_candidate(model_name, candidates, train_df, val_df, feature_cols):
    X_train = train_df[feature_cols]
    y_train = train_df[TARGET_COL]
    X_val = val_df[feature_cols]

    rebalance_dates = get_rebalance_dates(val_df)
    candidate_rows = []

    for candidate in candidates:
        estimator = clone(candidate["estimator"])
        estimator.fit(X_train, y_train)

        val_eval = val_df[["date", "code", REALIZED_RETURN_COL, TARGET_COL]].copy()
        val_eval["prediction"] = estimator.predict(X_val)

        bt, metrics = evaluate_signal(val_eval, "prediction", target_col=REALIZED_RETURN_COL, rebalance_dates=rebalance_dates)
        candidate_rows.append({
            "model": model_name,
            "candidate_label": candidate["label"],
            "validation_sharpe": metrics["sharpe_ratio"],
            "validation_cumulative_return": metrics["cumulative_return"],
            "validation_max_drawdown": metrics["max_drawdown"],
            "validation_average_turnover": metrics["average_turnover"],
            "estimator": deepcopy(candidate["estimator"]),
        })

    candidate_df = pd.DataFrame(candidate_rows).sort_values(
        ["validation_sharpe", "validation_cumulative_return", "validation_max_drawdown", "validation_average_turnover"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    best_row = candidate_df.iloc[0].to_dict()
    return best_row, candidate_df.drop(columns=["estimator"])


def fit_and_test_model(best_candidate, train_df, val_df, test_df, feature_cols):
    X_train_val = pd.concat([train_df[feature_cols], val_df[feature_cols]], axis=0)
    y_train_val = pd.concat([train_df[TARGET_COL], val_df[TARGET_COL]], axis=0)
    X_test = test_df[feature_cols]

    estimator = clone(best_candidate["estimator"])
    estimator.fit(X_train_val, y_train_val)

    test_eval = test_df[["date", "code", REALIZED_RETURN_COL, TARGET_COL]].copy()
    test_eval["prediction"] = estimator.predict(X_test)

    bt, metrics = evaluate_signal(test_eval, "prediction", target_col=REALIZED_RETURN_COL)
    return bt, metrics


def average_metric_dict(metric_rows):
    metric_df = pd.DataFrame(metric_rows)
    return {
        "cumulative_return": metric_df["cumulative_return"].mean(),
        "annualized_return": metric_df["annualized_return"].mean(),
        "annualized_volatility": metric_df["annualized_volatility"].mean(),
        "sharpe_ratio": metric_df["sharpe_ratio"].mean(),
        "max_drawdown": metric_df["max_drawdown"].mean(),
        "average_turnover": metric_df["average_turnover"].mean(),
        "n_periods": int(round(metric_df["n_periods"].mean())),
    }


def evaluate_baselines(test_df, test_year, n_random_seeds=50):
    rebalance_dates = get_rebalance_dates(test_df)
    random_metric_rows = []

    for seed_offset in range(n_random_seeds):
        seed = test_year * 1000 + seed_offset
        random_bt = long_short_backtest(
            test_df,
            build_random_long_short_weights(test_df, seed=seed, rebalance_dates=rebalance_dates),
            target_col=REALIZED_RETURN_COL,
        )
        random_metric_rows.append(backtest_metrics(random_bt))

    random_avg_metrics = average_metric_dict(random_metric_rows)
    return [
        build_strategy_result_row(
            test_year,
            f"Random Ranking Long-Short ({n_random_seeds}-Seed Average)",
            random_avg_metrics,
            "Baseline",
        )
    ]


def run_fold(spec, df_model, feature_cols):
    split = create_time_split(df_model, spec)
    train_df = split["train"]
    val_df = split["validation"]
    test_df = split["test"]

    if min(len(train_df), len(val_df), len(test_df)) == 0:
        raise ValueError(f"Empty split encountered for fold {spec['fold_name']}")

    fold_rows = []
    tuning_rows = []
    backtests = {}

    for model_name, candidates in MODEL_SPECS.items():
        best_candidate, candidate_df = select_best_model_candidate(model_name, candidates, train_df, val_df, feature_cols)
        bt, metrics = fit_and_test_model(best_candidate, train_df, val_df, test_df, feature_cols)
        category = "Baseline" if model_name == "Ridge / Linear Regression" else "ML Model"

        fold_rows.append({
            "test_year": spec["test_year"],
            "strategy": model_name,
            "category": category,
            "selected_candidate": best_candidate["candidate_label"],
            "validation_sharpe": best_candidate["validation_sharpe"],
            "validation_cumulative_return": best_candidate["validation_cumulative_return"],
            "validation_max_drawdown": best_candidate["validation_max_drawdown"],
            "validation_average_turnover": best_candidate["validation_average_turnover"],
            "net_cumulative_return": metrics["cumulative_return"],
            "sharpe_ratio": metrics["sharpe_ratio"],
            "max_drawdown": metrics["max_drawdown"],
            "average_turnover": metrics["average_turnover"],
            "n_rebalance_periods": metrics["n_periods"],
            "direction": 1,
            "train_start": spec["train_start"],
            "train_end": spec["train_end"],
            "val_start": spec["val_start"],
            "val_end": spec["val_end"],
            "test_start": spec["test_start"],
            "test_end": spec["test_end"],
        })

        tuning_table = candidate_df.copy()
        tuning_table["test_year"] = spec["test_year"]
        tuning_rows.append(tuning_table)
        backtests[model_name] = bt

    baseline_rows = evaluate_baselines(test_df, spec["test_year"])
    fold_rows.extend(baseline_rows)

    return {
        "fold_results": pd.DataFrame(fold_rows),
        "tuning_results": pd.concat(tuning_rows, ignore_index=True),
        "backtests": backtests,
    }


In [ ]:
all_fold_results = []
all_tuning_results = []
all_backtests = {}

for spec in FOLD_SPECS:
    print(
        f"{spec['fold_name']}: "
        f"train {spec['train_start']} to {spec['train_end']} | "
        f"validation {spec['val_start']} to {spec['val_end']} | "
        f"test {spec['test_start']} to {spec['test_end']}"
    )
    fold_output = run_fold(spec, model_df, feature_cols)
    all_fold_results.append(fold_output["fold_results"])
    all_tuning_results.append(fold_output["tuning_results"])
    all_backtests[spec["test_year"]] = fold_output["backtests"]

fold_results_df = pd.concat(all_fold_results, ignore_index=True)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

yearly_comparison_df = (
    fold_results_df
    .pivot(index="test_year", columns="strategy", values="net_cumulative_return")
    .sort_index()
    .reset_index()
)
yearly_comparison_df.columns.name = None

summary_table_df = (
    fold_results_df
    .groupby(["strategy", "category"], as_index=False)
    .agg(
        average_cumulative_return=("net_cumulative_return", "mean"),
        average_sharpe_ratio=("sharpe_ratio", "mean"),
        average_max_drawdown=("max_drawdown", "mean"),
        average_turnover=("average_turnover", "mean"),
        positive_test_folds=("net_cumulative_return", lambda x: int((x > 0).sum())),
    )
    .sort_values(["category", "average_sharpe_ratio"], ascending=[True, False])
    .reset_index(drop=True)
)

fold_results_csv = "rolling5y_fold_results_final.csv"
tuning_results_csv = "rolling5y_model_tuning_final.csv"
yearly_comparison_csv = "rolling5y_yearly_comparison_final.csv"
summary_table_csv = "rolling5y_summary_table_final.csv"

fold_results_df.to_csv(fold_results_csv, index=False)
tuning_results_df.to_csv(tuning_results_csv, index=False)
yearly_comparison_df.to_csv(yearly_comparison_csv, index=False)
summary_table_df.to_csv(summary_table_csv, index=False)

print("\nYearly Net Cumulative Return Comparison")
display(yearly_comparison_df.round(4))

print("\nSummary Table")
display(summary_table_df.round(4))

print("\nSaved CSV files:")
print(f"- {fold_results_csv}")
print(f"- {tuning_results_csv}")
print(f"- {yearly_comparison_csv}")
print(f"- {summary_table_csv}")

Fold 1: train 2011-01-01 to 2015-12-31 | validation 2016-01-01 to 2016-12-31 | test 2017-01-01 to 2017-12-31
